# DanceTrack val 전체 · Hybrid-SORT와 OA-SORT 재구현

Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 위에서부터 실행하세요. 기본 설정은 공식 `val_seqmap.txt`에 있는 **모든 시퀀스의 모든 프레임**을 평가합니다. 시퀀스마다 YOLOX 검출을 한 번 계산해 두 추적기에 공유하고, 공식 TrackEval의 `COMBINED`와 시퀀스별 지표를 저장합니다.

Hybrid-SORT는 기존 노트북과 같은 공식 데모의 DanceTrack 설정을 사용합니다. OA-SORT는 논문의 OAM·OAO·BAM 재구현입니다. 1차 매칭에 OAO를 적용하고, 저신뢰도 검출의 업데이트에 BAM을 적용합니다. DanceTrack 설정은 `tau=0.15`, Gaussian `(kx, ky)=(3√2, 3)`입니다. 박스 하단 차이 5픽셀과 기존 Hybrid-SORT의 최종 매칭 임계값은 이 재구현에서 사용한 설정입니다.

[DanceTrack 공식 평가](https://github.com/DanceTrack/DanceTrack#evaluation) · [Hybrid-SORT 공식 저장소](https://github.com/ymzis69/HybridSORT) · [OA-SORT 논문 표 11](https://arxiv.org/html/2603.06034v2)


## 1. 설정

`SMOKE_SEQUENCE = ""`를 유지하면 전체를 평가합니다. 처음에는 `"dancetrack0079"`로 실행을 점검할 수도 있습니다. 단일 영상 결과는 별도 폴더에 저장됩니다.

Google Drive에 이미 받은 데이터나 가중치가 있으면 `MOUNT_DRIVE = True`로 바꾼 뒤 경로를 지정하세요. 런타임이 초기화되어도 완료 결과를 이어서 쓰려면 `RESULTS_ROOT`도 Drive로 바꾸세요.


In [ ]:
from pathlib import Path

SMOKE_SEQUENCE = ""          # 기본은 val 전체. "dancetrack0079"는 단일 영상 점검
DANCETRACK_ZIP_PATH = ""      # 예: "/content/drive/MyDrive/DanceTrack/val.zip"
DANCETRACK_VAL_DIR = ""       # 이미 압축을 풀었다면 val 폴더의 전체 경로
MOUNT_DRIVE = False

WEIGHTS_MODE = "official"     # "official", "local", "drive_url", "bytetrack_hf"
WEIGHTS_PATH = ""             # 예: "/content/drive/MyDrive/ocsort_dance_model.pth.tar"
WEIGHTS_FILE_URL = ""         # drive_url 모드에서 공유된 '파일' 링크 입력
ALLOW_LEGACY_CHECKPOINT = False  # 신뢰하는 원본에서 weights_only 로딩 오류가 날 때만 True

USE_FP16 = True               # README의 DanceTrack 설정
REPO_REF = ""                 # 재실험할 때 출력된 Git 커밋 SHA를 넣을 수 있음
ROOT = Path("/content/hybrid_sort_colab")
RESULTS_ROOT = ROOT / "val_benchmarks"
# RESULTS_ROOT = Path("/content/drive/MyDrive/oa_sort_val_runs")
REPO = ROOT / "HybridSORT"
VENV = ROOT / "venv"
PYTHON = VENV / "bin/python"
VAL_CFG = "exps/example/mot/yolox_dancetrack_val_hybrid_sort.py"
CHECKPOINT_NAME = "ocsort_dance_model.pth.tar"
MODEL_ZOO_URLS = [
    "https://drive.google.com/drive/folders/18IsZGeGiyKDshhYIzbpYXoNEcBhPY8lN",
    "https://drive.google.com/drive/folders/1LnhZVJlpufUnWuObZASIN1KwfhuvT_a8",
]
VAL_ZIP_URL = "https://huggingface.co/datasets/noahcao/dancetrack/resolve/main/val.zip"
FALLBACK_WEIGHTS_URL = (
    "https://huggingface.co/noahcao/dancetrack_models/resolve/main/"
    "bytetrack_models/bytetrack_model.pth.tar"
)
ROOT.mkdir(parents=True, exist_ok=True)
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OA_TAU = 0.15                    # DanceTrack, Eq. (8)
OA_DEPTH_PIXELS = 5               # 박스의 아래쪽 y 차이, 원본 영상 픽셀
OA_GAUSSIAN = True


## 2. 실행 환경

Hybrid-SORT와 추적 의존성을 Python 3.11 가상환경에 설치하고 CUDA를 확인합니다.


In [ ]:
import os
import shutil
import subprocess
import sys

def run(args, *, cwd=None, env=None, capture=False):
    result = subprocess.run(
        [str(arg) for arg in args], cwd=cwd, env=env, check=True, text=True,
        stdout=subprocess.PIPE if capture else None,
    )
    return result.stdout.strip() if capture else None

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Colab 런타임 유형에서 GPU(T4)를 선택한 후 다시 실행하세요.")
run(["nvidia-smi", "-L"])

if not (REPO / ".git").is_dir():
    run(["git", "clone", "https://github.com/ymzis69/HybridSORT.git", REPO])
if REPO_REF:
    run(["git", "checkout", "--detach", REPO_REF], cwd=REPO)
REVISION = run(["git", "rev-parse", "HEAD"], cwd=REPO, capture=True)
print("Hybrid-SORT commit:", REVISION)

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
uv = shutil.which("uv") or str(Path(sys.executable).with_name("uv"))
bootstrap = VENV / ".hybrid-sort-env-val-v1"
if not bootstrap.is_file() or not PYTHON.is_file():
    run([uv, "python", "install", "3.11"])
    if not PYTHON.exists():
        run([uv, "venv", "--python", "3.11", VENV])
    install = [uv, "pip", "install", "--python", PYTHON]
    run([*install, "--index-url", "https://download.pytorch.org/whl/cu124",
         "torch==2.6.0", "torchvision==0.21.0"])
    run([*install,
         "numpy==1.23.5", "scipy==1.13.1", "opencv-python-headless==4.10.0.84",
         "lap==0.5.12", "filterpy==1.4.5", "scikit-image==0.22.0",
         "pandas==2.2.3", "matplotlib==3.7.5", "scikit-learn==1.5.2",
         "cython==0.29.37", "setuptools==69.5.1", "wheel",
         "loguru", "tqdm", "pillow", "thop", "ninja", "tabulate", "tensorboard",
         "h5py", "xmltodict", "yacs", "easydict", "termcolor", "fvcore",
         "iopath", "pyyaml", "prettytable", "einops", "faiss-cpu==1.8.0.post1", "gdown==6.1.0"])
    run([*install, "--no-build-isolation", "cython-bbox==0.1.5", "pycocotools==2.0.7"])
    bootstrap.touch()

BASE_ENV = {**os.environ, "PYTHONPATH": str(REPO), "MPLBACKEND": "Agg",
            "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "YOLOX_DATADIR": str(REPO / "datasets")}
run([PYTHON, "-c", "import cv2, numpy, torch, torchvision, lap, cython_bbox; "
     "assert torch.cuda.is_available(), 'CUDA를 사용할 수 없습니다. GPU 런타임을 확인하세요.'; "
     "print('GPU:', torch.cuda.get_device_name()); "
     "print('torch / numpy:', torch.__version__, numpy.__version__)"], cwd=REPO, env=BASE_ENV)


## 3. 공통 YOLOX 가중치

기본값 `WEIGHTS_MODE = "official"`은 Hybrid-SORT README에 적힌 `ocsort_dance_model.pth.tar`를 가져옵니다. Google Drive 다운로드 제한이 걸리면 모델 파일을 직접 받아 `WEIGHTS_MODE = "local"`과 `WEIGHTS_PATH`를 지정하세요. `"bytetrack_hf"`는 다른 검출기 가중치로 비교할 때 사용할 수 있습니다.


In [ ]:
import hashlib
import urllib.request

PRETRAINED = REPO / "pretrained"
PRETRAINED.mkdir(exist_ok=True)

def fetch_http(url, target):
    partial = target.with_name(target.name + ".part")
    with urllib.request.urlopen(url, timeout=60) as response, partial.open("wb") as output:
        shutil.copyfileobj(response, output, length=1024 * 1024)
    partial.replace(target)

if WEIGHTS_MODE == "local":
    if not WEIGHTS_PATH:
        raise ValueError("1번 셀에 WEIGHTS_PATH를 지정하세요.")
    checkpoint = Path(WEIGHTS_PATH).expanduser()
elif WEIGHTS_MODE == "bytetrack_hf":
    checkpoint = PRETRAINED / "bytetrack_model.pth.tar"
    if not checkpoint.is_file() or checkpoint.stat().st_size < 1_000_000:
        fetch_http(FALLBACK_WEIGHTS_URL, checkpoint)
elif WEIGHTS_MODE in {"official", "drive_url"}:
    # 직접 지정한 다른 파일 링크는 공식 체크포인트 캐시와 구분한다.
    url_key = hashlib.sha256(WEIGHTS_FILE_URL.encode()).hexdigest()[:12]
    checkpoint = PRETRAINED / (CHECKPOINT_NAME if WEIGHTS_MODE == "official"
                               else f"drive-{url_key}.pth.tar")
    if WEIGHTS_MODE == "drive_url" and not WEIGHTS_FILE_URL:
        raise ValueError("1번 셀에 WEIGHTS_FILE_URL을 지정하세요.")
    if not checkpoint.is_file() or checkpoint.stat().st_size < 1_000_000:
        download_code = r'''
import pathlib, sys
import gdown

target = pathlib.Path(sys.argv[1])
name, direct_url, *folders = sys.argv[2:]
temporary = target.with_name(target.name + ".part")
errors = []
if not direct_url:
    for folder in folders:
        try:
            entries = gdown.download_folder(url=folder, skip_download=True)
            entry = next((item for item in entries if pathlib.Path(item.path).name == name), None)
            if entry is None:
                raise FileNotFoundError(name)
            result = gdown.download(id=entry.id, output=str(temporary))
            if not result or temporary.stat().st_size < 1_000_000:
                raise OSError("가중치 다운로드가 완료되지 않았습니다.")
            temporary.replace(target)
            break
        except Exception as exc:
            errors.append(f"{folder}: {exc}")
    else:
        raise SystemExit("공식 가중치 자동 다운로드 실패. 파일을 직접 받은 뒤 local 모드를 사용하세요.\n"
                         + "\n".join(errors))
else:
    result = gdown.download(url=direct_url, output=str(temporary))
    if not result or temporary.stat().st_size < 1_000_000:
        raise SystemExit("Drive 링크로 가중치를 받지 못했습니다. local 모드를 사용하세요.")
    temporary.replace(target)
'''
        run([PYTHON, "-c", download_code, checkpoint, CHECKPOINT_NAME,
             WEIGHTS_FILE_URL if WEIGHTS_MODE == "drive_url" else "", *MODEL_ZOO_URLS],
            cwd=REPO, env=BASE_ENV)
else:
    raise ValueError(f"지원하지 않는 WEIGHTS_MODE: {WEIGHTS_MODE}")

if not checkpoint.is_file() or checkpoint.stat().st_size < 1_000_000:
    raise FileNotFoundError(f"가중치 파일을 확인하세요: {checkpoint}")
checkpoint = checkpoint.resolve()
with checkpoint.open("rb") as file:
    checkpoint_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
print(f"YOLOX checkpoint: {checkpoint} ({checkpoint.stat().st_size / 1e6:.1f} MB)")
print("SHA-256:", checkpoint_sha256)


## 4. 검증 세트 준비

경로를 비워두면 Hugging Face에서 공식 `val.zip`(약 4.2 GB)을 받습니다. 압축 파일에서 시퀀스 목록과 각 `seqinfo.ini`의 프레임 수를 확인하고, 실제 추적할 때는 시퀀스 하나씩 임시로 풉니다.


In [ ]:
import configparser
import re
from collections import defaultdict
from zipfile import ZipFile, is_zipfile

SEQMAP_URL = (
    "https://raw.githubusercontent.com/DanceTrack/DanceTrack/main/dancetrack/val_seqmap.txt"
)
CACHE = ROOT / "dancetrack_val_cache"
CACHE.mkdir(parents=True, exist_ok=True)
official_map = CACHE / "val_seqmap.txt"
if not official_map.is_file():
    fetch_http(SEQMAP_URL, official_map)

split_lines = [line.strip() for line in official_map.read_text(encoding="utf-8-sig").splitlines()
               if line.strip()]
if split_lines and split_lines[0].lower() == "name":
    split_lines = split_lines[1:]
if (not split_lines or len(set(split_lines)) != len(split_lines)
        or any(not re.fullmatch(r"dancetrack\d{4}", name) for name in split_lines)):
    raise ValueError(f"공식 val 시퀀스 목록의 형식을 확인하세요. {official_map}")
if SMOKE_SEQUENCE and SMOKE_SEQUENCE not in split_lines:
    raise ValueError(f"SMOKE_SEQUENCE는 공식 val에 포함된 시퀀스여야 합니다. {SMOKE_SEQUENCE}")
SEQUENCES = [SMOKE_SEQUENCE] if SMOKE_SEQUENCE else split_lines
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

if DANCETRACK_VAL_DIR and DANCETRACK_ZIP_PATH:
    raise ValueError("DANCETRACK_VAL_DIR와 DANCETRACK_ZIP_PATH 중 하나만 지정하세요.")
VAL_DIR = Path(DANCETRACK_VAL_DIR).expanduser().resolve() if DANCETRACK_VAL_DIR else None
ZIP_PATH = None
if VAL_DIR is None:
    ZIP_PATH = (Path(DANCETRACK_ZIP_PATH).expanduser().resolve() if DANCETRACK_ZIP_PATH
                else CACHE / "val.zip")
    if not ZIP_PATH.is_file():
        if DANCETRACK_ZIP_PATH:
            raise FileNotFoundError(ZIP_PATH)
        partial = ZIP_PATH.with_suffix(".zip.part")
        if not is_zipfile(partial):
            existing = partial.stat().st_size if partial.exists() else 0
            if shutil.disk_usage(CACHE).free + existing < 6e9:
                raise OSError("자동 다운로드와 임시 프레임에 필요한 디스크 공간이 부족합니다.")
            run(["curl", "--fail", "--location", "--retry", "3", "--continue-at", "-",
                 "--output", partial, VAL_ZIP_URL])
        with ZipFile(partial):
            pass
        partial.replace(ZIP_PATH)


def digest_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def ordered_frames(names, sequence, expected):
    ordered = sorted(names)
    if (len(ordered) != expected or any(not Path(name).stem.isdecimal()
            or int(Path(name).stem) != index for index, name in enumerate(ordered, 1))):
        raise ValueError(f"{sequence}: seqinfo 기준 1–{expected} 프레임 파일이 필요합니다.")
    return ordered


def sequence_metadata(sequence, seqinfo):
    metadata = configparser.ConfigParser()
    metadata.read_string(seqinfo.decode("utf-8-sig"))
    section = metadata["Sequence"]
    length = section.getint("seqLength")
    if length < 1:
        raise ValueError(f"{sequence}: seqLength 오류")
    return length


def inspect_sequences(names, val_dir, zip_path):
    """Read GT and file inventories before spending time on detector inference."""
    if val_dir is not None:
        data = {}
        for name in names:
            folder = val_dir / name
            seqinfo = (folder / "seqinfo.ini").read_bytes()
            ground_truth = (folder / "gt" / "gt.txt").read_bytes()
            image_folder = folder / "img1"
            frames = ordered_frames(
                [file.name for file in image_folder.iterdir()
                 if file.is_file() and file.suffix.lower() in IMAGE_SUFFIXES],
                name, sequence_metadata(name, seqinfo),
            )
            inventory = []
            for image in frames:
                stat = (image_folder / image).stat()
                inventory.append((image, stat.st_size, stat.st_mtime_ns))
            data[name] = dict(info=seqinfo, gt=ground_truth, frames=frames, inventory=inventory)
        return data

    selected = set(names)
    folders = defaultdict(dict)
    with ZipFile(zip_path) as archive:
        for entry in archive.infolist():
            if entry.is_dir():
                continue
            parts = entry.filename.split("/")
            for index, part in enumerate(parts[:-1]):
                if part not in selected or "__MACOSX" in parts[:index]:
                    continue
                tail = tuple(parts[index + 1:])
                if tail in {("seqinfo.ini",), ("gt", "gt.txt")} or (
                    len(tail) == 2 and tail[0] == "img1"
                    and Path(tail[1]).suffix.lower() in IMAGE_SUFFIXES
                    and Path(tail[1]).stem.isdecimal()
                ):
                    folder = folders[(part, tuple(parts[:index]))]
                    if tail in folder:
                        raise ValueError(f"ZIP에 중복 항목이 있습니다. {entry.filename}")
                    folder[tail] = entry
                break

        data = {}
        for name in names:
            candidates = [members for (sequence, _prefix), members in folders.items()
                          if sequence == name and ("seqinfo.ini",) in members
                          and ("gt", "gt.txt") in members]
            if len(candidates) != 1:
                raise ValueError(f"{name}: val.zip에서 seqinfo.ini와 gt/gt.txt를 확인하세요.")
            members = candidates[0]
            seqinfo = archive.read(members[("seqinfo.ini",)])
            ground_truth = archive.read(members[("gt", "gt.txt")])
            frames = ordered_frames(
                [tail[1] for tail in members if tail[0] == "img1"],
                name, sequence_metadata(name, seqinfo),
            )
            entries = [members[("img1", frame)] for frame in frames]
            data[name] = dict(
                info=seqinfo, gt=ground_truth, frames=frames,
                entries=[entry.filename for entry in entries],
                inventory=[(frame, entry.file_size, entry.CRC) for frame, entry in zip(frames, entries)],
            )
    return data


DATA = inspect_sequences(SEQUENCES, VAL_DIR, ZIP_PATH)
for name, details in DATA.items():
    if not details["gt"].strip():
        raise ValueError(f"{name}: 정답 gt.txt가 비어 있습니다.")
print(f"{'단일 시퀀스 점검' if SMOKE_SEQUENCE else 'DanceTrack val 전체'} | "
      f"{len(SEQUENCES)}/{len(split_lines)}개 시퀀스 | {sum(len(d['frames']) for d in DATA.values()):,}프레임")


## 5. 추적 및 평가 코드 준비

논문 기반 OA-SORT 모듈과 공통 검출·추적 실행 파일을 생성합니다.


In [ ]:
# 5. OA-SORT 구현 파일 생성
from pathlib import Path

CODE_DIR = ROOT / "val_source"
CODE_DIR.mkdir(parents=True, exist_ok=True)

OA_SOURCE = r'''
"""Paper reimplementation of OA-SORT on top of upstream Hybrid-SORT."""
import ast
from collections import Counter
from dataclasses import dataclass
import inspect
import math
from textwrap import dedent

import numpy as np


@dataclass(frozen=True)
class OAConfig:
    use_oao: bool = True
    use_bam: bool = True
    gaussian: bool = True
    tau: float = 0.15
    depth_pixels: float = 5.0
    kx: float = 3 * math.sqrt(2)
    ky: float = 3.0

    def __post_init__(self):
        if not all(math.isfinite(value) for value in (self.tau, self.depth_pixels, self.kx, self.ky)):
            raise ValueError("OA-SORT parameters must be finite.")
        if not 0 <= self.tau <= 1 or min(self.depth_pixels, self.kx, self.ky) <= 0:
            raise ValueError("Require 0 <= tau <= 1 and positive depth_pixels, kx and ky.")


def occlusion_coefficients(boxes, image_hw, config):
    """OAM (Eq. 2–7). Boxes are N×4 xyxy in original image pixels."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    coefficients = np.zeros(len(boxes), dtype=np.float32)
    height, width = map(int, image_hw)
    if len(boxes) < 2 or min(height, width) <= 0:
        return coefficients

    finite = np.isfinite(boxes).all(axis=1)
    # Use floor/ceil for the image slices and floating-point bottom edges for depth.
    clipped = np.clip(np.where(finite[:, None], boxes, 0), 0, [width, height, width, height])
    starts = np.floor(clipped[:, :2]).astype(np.intp)
    ends = np.ceil(clipped[:, 2:]).astype(np.intp)
    valid = finite & (clipped[:, 2:] > clipped[:, :2]).all(axis=1)
    overlap_start = np.maximum(starts[:, None, :], starts[None, :, :])
    overlap_end = np.minimum(ends[:, None, :], ends[None, :, :])
    overlap = (overlap_end > overlap_start).all(axis=2) & valid[:, None] & valid[None, :]
    # Suppress pixel quantization overlaps between geometrically disjoint boxes.
    overlap &= (np.minimum(clipped[:, None, 2:], clipped[None, :, 2:])
                > np.maximum(clipped[:, None, :2], clipped[None, :, :2])).all(axis=2)
    np.fill_diagonal(overlap, False)
    bottoms = clipped[:, 3]
    occludes = overlap & ((bottoms[None, :] - bottoms[:, None]) >= config.depth_pixels)
    blocked = np.flatnonzero(occludes.any(axis=1))
    if not len(blocked):
        return coefficients

    if config.gaussian:
        gaussian_map = np.zeros((height, width), dtype=np.float32)
        for index in np.flatnonzero(overlap.any(axis=1)):
            left, top = starts[index]
            right, bottom = ends[index]
            center = (clipped[index, :2] + clipped[index, 2:]) * 0.5
            sigma = (clipped[index, 2:] - clipped[index, :2]) / [config.kx, config.ky]
            dx = (np.arange(left, right) + 0.5 - center[0]) / sigma[0]
            dy = (np.arange(top, bottom) + 0.5 - center[1]) / sigma[1]
            weights = np.exp(-0.5 * (dy[:, None] ** 2 + dx[None, :] ** 2))
            region = gaussian_map[top:bottom, left:right]
            np.maximum(region, weights, out=region)

    for index in blocked:
        left, top = starts[index]
        right, bottom = ends[index]
        mask = np.zeros((bottom - top, right - left), dtype=bool)
        for other in np.flatnonzero(occludes[index]):
            x1, y1 = overlap_start[index, other]
            x2, y2 = overlap_end[index, other]
            mask[y1 - top:y2 - top, x1 - left:x2 - left] = True
        covered = (gaussian_map[top:bottom, left:right][mask].sum(dtype=np.float64)
                   if config.gaussian else mask.sum())
        coefficients[index] = covered / mask.size
    return coefficients


def occlusion_aware_similarity(similarity, predicted_occlusion, tau):
    """OAO (Eq. 8). Columns represent predicted tracks."""
    return (1 - tau) * similarity + tau * (1 - predicted_occlusion[None, :])


def bias_aware_weight(spatial_similarity, last_observed_occlusion):
    """BAM (Eq. 9), with HMIoU following the paper's Hybrid-SORT experiment."""
    return float(np.clip(spatial_similarity, 0, 1) * (1 - last_observed_occlusion))


def damp_geometry_residual(residual, weight):
    # Hybrid-SORT measurement layout: x, y, area, confidence, aspect ratio.
    if residual.shape[0] != 5:
        raise RuntimeError(f"Expected the 5D Hybrid-SORT measurement, got {residual.shape}.")
    residual[[0, 1, 2, 4]] *= weight


def method_ast(function):
    return ast.parse(dedent(inspect.getsource(function)))


def compile_method(tree, original, **symbols):
    namespace = {**original.__globals__, **symbols}
    ast.fix_missing_locations(tree)
    exec(compile(tree, f"<OA integration: {original.__qualname__}>", "exec"), namespace)
    return namespace[original.__name__]


def build_kf_update(original):
    """Insert BAM after the KF innovation is computed, including after ORU."""
    tree = method_ast(original)
    method = tree.body[0]
    residuals = [
        index for index, node in enumerate(method.body)
        if isinstance(node, ast.Assign)
        and any(ast.unparse(target) == "self.y" for target in node.targets)
        and isinstance(node.value, ast.BinOp) and isinstance(node.value.op, ast.Sub)
    ]
    if len(residuals) != 1:
        raise RuntimeError("Hybrid-SORT KF update changed. Could not locate the innovation.")
    method.args.kwonlyargs.append(ast.arg(arg="oa_weight"))
    method.args.kw_defaults.append(None)
    method.body.insert(residuals[0] + 1, ast.parse("damp_geometry_residual(self.y, oa_weight)").body[0])
    return compile_method(tree, original, damp_geometry_residual=damp_geometry_residual)


def build_track_update(original, kf_update):
    """Keep the raw observation in Hybrid-SORT's motion and confidence histories."""
    tree = method_ast(original)
    method = tree.body[0]
    locations = [
        node for node in ast.walk(tree) if isinstance(node, ast.Call)
        and ast.unparse(node.func) == "self.kf.update" and len(node.args) == 1
        and ast.unparse(node.args[0]) == "convert_bbox_to_z(bbox)"
    ]
    if len(locations) != 1:
        raise RuntimeError("Hybrid-SORT track update changed. Could not attach BAM.")
    method.args.kwonlyargs.append(ast.arg(arg="oa_weight"))
    method.args.kw_defaults.append(None)
    target = locations[0]
    target.func = ast.Name(id="oa_kf_update", ctx=ast.Load())
    target.args.insert(0, ast.parse("self.kf", mode="eval").body)
    target.keywords.append(ast.keyword(arg="oa_weight", value=ast.Name(id="oa_weight", ctx=ast.Load())))
    return compile_method(tree, original, oa_kf_update=kf_update)


def build_tracker_update(original):
    """Attach OAO at stage 1 and BAM at accepted stage-2 matches only."""
    tree = method_ast(original)
    high_calls = 0
    low_calls = 0
    for node in list(ast.walk(tree)):
        if not isinstance(node, ast.Call):
            continue
        called = ast.unparse(node.func)
        if called in {"associate_4_points_with_score", "associate_4_points"}:
            node.args.insert(0, node.func)
            node.func = ast.parse("self._associate_high", mode="eval").body
            high_calls += 1
        elif (called == "self.trackers[trk_ind].update" and node.args
              and ast.unparse(node.args[0]).startswith("dets_second[")):
            node.func = ast.parse("self._update_low", mode="eval").body
            node.args.insert(0, ast.Name(id="trk_ind", ctx=ast.Load()))
            node.args.append(ast.parse("trks[trk_ind, :4]", mode="eval").body)
            low_calls += 1
    if (high_calls, low_calls) != (2, 1):
        raise RuntimeError(
            f"Hybrid-SORT update changed: found {high_calls} high and {low_calls} low hooks; expected 2 and 1."
        )
    return compile_method(tree, original)


def calls_linear_assignment(nodes):
    return any(
        isinstance(child, ast.Call)
        and isinstance(child.func, ast.Name)
        and child.func.id == "linear_assignment"
        for node in nodes
        for child in ast.walk(node)
    )


def build_high_association(original):
    """Apply Algorithm 4 even on the upstream one-to-one fast path."""
    tree = method_ast(original)
    method = tree.body[0]
    if not calls_linear_assignment(method.body):
        raise RuntimeError("Hybrid-SORT stage 1 no longer calls linear_assignment.")

    method.args.kwonlyargs.append(ast.arg(arg="oa_adjust_cost"))
    method.args.kw_defaults.append(None)
    for node in list(ast.walk(method)):
        if isinstance(node, ast.If) and calls_linear_assignment(node.orelse):
            if not calls_linear_assignment(node.body):
                if any(
                    isinstance(child, ast.Call) and ast.unparse(child.func) == "np.where"
                    for statement in node.body
                    for child in ast.walk(statement)
                ):
                    # Reuse the branch which combines spatial, direction and
                    # confidence scores. Keep the upstream final IoU gate.
                    node.test = ast.Constant(value=False)
        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Name)
                and node.func.id == "linear_assignment"):
            if len(node.args) != 1 or node.keywords:
                raise RuntimeError("Unexpected signature of stage-1 linear_assignment.")
            node.args[0] = ast.Call(
                func=ast.Name(id="oa_adjust_cost", ctx=ast.Load()),
                args=[node.args[0]],
                keywords=[],
            )
    return compile_method(tree, original)


def make_oa_tracker(base, kf_class):
    """Construct a subclass using the installed Hybrid-SORT implementation."""
    original_update = build_tracker_update(base.Hybrid_Sort.update)
    bam_track_update = build_track_update(
        base.KalmanBoxTracker.update, build_kf_update(kf_class.update)
    )
    high_associations = {
        function: build_high_association(function)
        for function in (
            base.Hybrid_Sort.update.__globals__["associate_4_points_with_score"],
            base.Hybrid_Sort.update.__globals__["associate_4_points"],
        )
    }
    spatial_hmiou = base.ASSO_FUNCS["Height_Modulated_IoU"]

    class OASort(base.Hybrid_Sort):
        def __init__(self, *args, oa_config, **kwargs):
            super().__init__(*args, **kwargs)
            self.oa_config = oa_config
            self.oa_stats = Counter()
            if oa_config.use_bam and not self.use_byte:
                raise RuntimeError("BAM needs Hybrid-SORT's low-score matching (use_byte=True).")

        def update(self, detections, img_info, img_size):
            self.image_hw = img_info[:2]
            tracks = original_update(self, detections, img_info, img_size)
            if self.oa_config.use_bam:
                # Algorithm 5 uses matched tracks. A newly initialized track still
                # has Hybrid-SORT's negative-score last_observation sentinel.
                observed = [track for track in self.trackers if track.time_since_update == 0
                            and track.last_observation[4] >= 0]
                boxes = [track.last_observation[:4] for track in observed]
                coefficients = occlusion_coefficients(boxes, self.image_hw, self.oa_config)
                for track, coefficient in zip(observed, coefficients):
                    track.oa_last_observed = float(coefficient)
                self.oa_stats["observed_occluded_tracks"] += int(np.count_nonzero(coefficients))
            return tracks

        def _associate_high(self, original, detections, predictions, *args, **kwargs):
            config = self.oa_config
            if not config.use_oao or config.tau == 0 or not len(detections) or not len(predictions):
                return original(detections, predictions, *args, **kwargs)

            coefficients = occlusion_coefficients(predictions[:, :4], self.image_hw, config)
            raw = self.asso_func(detections, predictions)
            adjusted = occlusion_aware_similarity(raw, coefficients, config.tau)
            self.oa_stats["predicted_occluded_tracks"] += int(np.count_nonzero(coefficients))
            def adjust_cost(cost):
                if cost.shape != raw.shape:
                    raise RuntimeError("Unexpected shape of the stage-1 assignment matrix.")
                self.oa_stats["oao_solver_calls"] += 1
                # The solver minimizes cost. Replace the spatial component
                # while retaining the direction and confidence terms.
                return cost + raw - adjusted

            previous_calls = self.oa_stats["oao_solver_calls"]
            matches = high_associations[original](
                detections, predictions, *args, **kwargs, oa_adjust_cost=adjust_cost
            )
            if self.oa_stats["oao_solver_calls"] != previous_calls + 1:
                raise RuntimeError("OAO expected one stage-1 assignment for nonempty inputs.")
            return matches

        def _update_low(self, track_index, detection, prediction):
            track = self.trackers[track_index]
            if not self.oa_config.use_bam:
                track.update(detection)
                return

            similarity = spatial_hmiou(detection[None, :4], prediction[None, :4])[0, 0]
            weight = bias_aware_weight(similarity, getattr(track, "oa_last_observed", 0.0))
            bam_track_update(track, detection, oa_weight=weight)
            self.oa_stats["bam_updates"] += 1
            if weight < 1:
                self.oa_stats["bam_damped_updates"] += 1

    return OASort
'''
(CODE_DIR / "oa_sort_impl.py").write_text(OA_SOURCE, encoding="utf-8")
print("OA-SORT 구현 생성:", CODE_DIR / "oa_sort_impl.py")

RUNNER_SOURCE = r'''
"""Use the official demo to capture one set of detections, then replay it."""
import argparse
import ast
from collections import Counter
from dataclasses import replace
import importlib
import inspect
import json
from pathlib import Path
import sys

import cv2
import numpy as np

from oa_sort_impl import OAConfig, make_oa_tracker


MODE_FLAGS = {"hybrid": (False, False), "oao_only": (True, False),
              "bam_only": (False, True), "oa_sort": (True, True)}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def model_detections(outputs, ratio):
    """Return [x1,y1,x2,y2,score] per detection in original image pixels."""
    candidates = outputs[0]
    if candidates is None:
        return np.empty((0, 5), dtype=np.float32)
    if hasattr(candidates, "detach"):
        candidates = candidates.detach().float().cpu().numpy()
    candidates = np.asarray(candidates, dtype=np.float32)
    if candidates.ndim != 2 or candidates.shape[1] < 5:
        raise ValueError(f"Unexpected YOLOX detection tensor: {candidates.shape}")
    boxes = candidates[:, :5].copy()
    boxes[:, :4] /= ratio
    if candidates.shape[1] > 5:
        boxes[:, 4] *= candidates[:, 5]
    return boxes


class DetectionCapture:
    def __init__(self, frames):
        self.frames = frames
        self.detections = []
        self.sizes = []
        self.tracker_spec = None

    def install(self, namespace):
        predictor_class = namespace.get("Predictor")
        upstream_tracker = namespace.get("Hybrid_Sort")
        if predictor_class is None or upstream_tracker is None:
            raise RuntimeError("Unexpected Hybrid-SORT demo API: expected Predictor and Hybrid_Sort.")

        def create_tracker(*args, **kwargs):
            if self.tracker_spec is not None:
                raise RuntimeError("Expected one Hybrid-SORT tracker in the official image demo.")
            self.tracker_spec = args, kwargs
            return DetectionOnlyTracker()

        original_inference = predictor_class.inference

        def capture_inference(predictor, image, *args, **kwargs):
            result = original_inference(predictor, image, *args, **kwargs)
            outputs, info = result[:2]
            index = len(self.detections)
            if index >= len(self.frames):
                raise RuntimeError("The demo received more images than expected.")
            if isinstance(image, (str, Path)) and Path(image).resolve() != self.frames[index]:
                raise RuntimeError(f"Unexpected input order at frame {index + 1}: {image}")
            if getattr(predictor, "test_size", None) is not None:
                test_h, test_w = predictor.test_size
                ratio = min(test_h / info["height"], test_w / info["width"])
            else:
                ratio = float(info["ratio"])
            if not np.isfinite(ratio) or ratio <= 0:
                raise RuntimeError(f"Invalid detector scale: {ratio}")
            self.detections.append(model_detections(outputs, ratio))
            self.sizes.append((int(info["height"]), int(info["width"])))
            return result

        predictor_class.inference = capture_inference
        namespace["Hybrid_Sort"] = create_tracker


class DetectionOnlyTracker:
    def update(self, *_args, **_kwargs):
        return np.empty((0, 5), dtype=np.float32)


def collect_detections(repo, frames, weights, fp16, exp_config, demo_output):
    """Enter the official demo with its original CLI, config and checkpoint loader."""
    checkpoint_path = weights.expanduser().resolve(strict=True)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint is not a file: {checkpoint_path}")

    script = repo / "tools/demo_track.py"
    tree = ast.parse(script.read_text(encoding="utf-8"), filename=str(script))

    # Older FastReID uses collections.Mapping. Import it only when requested.
    lazy_reid = ast.parse(
        "def FastReIDInterface(*args, **kwargs):\n"
        "    from fast_reid.fast_reid_interfece import FastReIDInterface as Interface\n"
        "    return Interface(*args, **kwargs)\n"
    ).body[0]
    for index, node in enumerate(tree.body):
        if (isinstance(node, ast.ImportFrom)
                and node.module == "fast_reid.fast_reid_interfece"
                and [(alias.name, alias.asname) for alias in node.names]
                    == [("FastReIDInterface", None)]):
            tree.body[index] = ast.copy_location(lazy_reid, node)
            break

    guards = [
        node for node in tree.body
        if isinstance(node, ast.If) and isinstance(node.test, ast.Compare)
        and isinstance(node.test.left, ast.Name) and node.test.left.id == "__name__"
        and any(isinstance(value, ast.Constant) and value.value == "__main__"
                for value in node.test.comparators)
    ]
    if len(guards) != 1:
        raise RuntimeError("Could not locate the official Hybrid-SORT demo entry point.")
    entry = guards[0].body
    main_calls = [
        index for index, stmt in enumerate(entry)
        if isinstance(stmt, ast.Expr) and isinstance(stmt.value, ast.Call)
        and isinstance(stmt.value.func, ast.Name) and stmt.value.func.id == "main"
    ]
    if len(main_calls) != 1:
        raise RuntimeError("Could not locate the Hybrid-SORT demo main() call.")

    # The experiment config can override the CLI checkpoint before main().
    entry[main_calls[0]:main_calls[0]] = ast.parse(
        "args.ckpt = oa_checkpoint_path\nargs.output_dir = oa_demo_output"
    ).body
    entry.insert(0, ast.parse("install_detection_capture(globals())").body[0])

    # Bind the actual load to the same file even if main() rewrites ckpt_file.
    demo_main, = [node for node in tree.body
                  if isinstance(node, ast.FunctionDef) and node.name == "main"]
    checkpoint_loads = [
        node for node in ast.walk(demo_main)
        if isinstance(node, ast.Call) and ast.unparse(node.func) == "torch.load"
        and node.args and isinstance(node.args[0], ast.Name)
        and node.args[0].id == "ckpt_file"
    ]
    if len(checkpoint_loads) != 1:
        raise RuntimeError("Could not locate the Hybrid-SORT checkpoint load.")
    checkpoint_loads[0].args[0] = ast.parse("oa_checkpoint_path", mode="eval").body
    ast.fix_missing_locations(tree)

    capture = DetectionCapture(frames)
    original_argv = sys.argv
    original_path = sys.path.copy()
    # OpenCV GUI calls are unavailable in Colab.
    old_imwrite = cv2.imwrite
    cv2.imwrite = lambda *_args, **_kwargs: True
    cv2.namedWindow = lambda *_args, **_kwargs: None
    cv2.imshow = lambda *_args, **_kwargs: None
    cv2.waitKey = lambda *_args, **_kwargs: -1
    cv2.destroyAllWindows = lambda *_args, **_kwargs: None
    # Upstream main initializes vis_folder only when --save_result is set.
    sys.argv = [str(script), "--demo_type", "image", "-f", str(exp_config),
                "-c", str(checkpoint_path), "--path", str(frames[0].parent), "--fuse", "--save_result"]
    if fp16:
        sys.argv.append("--fp16")
    sys.path.insert(0, str(script.parent))
    try:
        namespace = {"__name__": "__main__", "__file__": str(script),
                     "install_detection_capture": capture.install,
                     "oa_checkpoint_path": str(checkpoint_path),
                     "oa_demo_output": str(demo_output)}
        print(f"[OA-SORT] Detector checkpoint: {checkpoint_path}", flush=True)
        exec(compile(tree, str(script), "exec"), namespace)
    finally:
        cv2.imwrite = old_imwrite
        sys.argv = original_argv
        sys.path[:] = original_path
    if capture.tracker_spec is None or len(capture.detections) != len(frames):
        raise RuntimeError(
            f"Expected {len(frames)} captured frames and one tracker config, "
            f"got {len(capture.detections)} frames. Check the upstream demo log."
        )
    return capture.detections, capture.sizes, capture.tracker_spec


def record_frame(tracks, frame_number, text_file, min_box_area):
    """Write results in the DanceTrack MOT format."""
    rows = 0
    for x1, y1, x2, y2, track_id in tracks:
        width, height = x2 - x1, y2 - y1
        if not np.isfinite([x1, y1, x2, y2, track_id]).all():
            continue
        if width <= 0 or height <= 0 or width * height <= min_box_area:
            continue
        text_file.write(f"{frame_number},{int(track_id)},{x1:.4f},{y1:.4f},"
                        f"{width:.4f},{height:.4f},1,-1,-1,-1\n")
        rows += 1
    return rows


def replay(image_sizes, detections, tracker_spec, base, kf_class, modes, config, dest, sequence):
    tracker_args, tracker_kwargs = tracker_spec
    arguments = inspect.signature(base.Hybrid_Sort).bind(*tracker_args, **tracker_kwargs)
    arguments.apply_defaults()
    # Eq. 8 and Eq. 9 use HMIoU for this configuration. Both sides of the
    # comparison also need the same low-score association.
    arguments.arguments["asso_func"] = "Height_Modulated_IoU"
    arguments.arguments["use_byte"] = True
    tracker_args, tracker_kwargs = arguments.args, arguments.kwargs
    oa_tracker = make_oa_tracker(base, kf_class) if any(mode != "hybrid" for mode in modes) else None
    summary = {}

    for mode in modes:
        use_oao, use_bam = MODE_FLAGS[mode]
        if mode == "hybrid":
            tracker = base.Hybrid_Sort(*tracker_args, **tracker_kwargs)
        else:
            tracker = oa_tracker(
                *tracker_args, **tracker_kwargs,
                oa_config=replace(config, use_oao=use_oao, use_bam=use_bam),
            )
        min_box_area = float(getattr(tracker.args, "min_box_area", 0))
        mode_dir = dest / mode
        mode_dir.mkdir(parents=True, exist_ok=True)
        total_rows = 0
        with (mode_dir / f"{sequence}.txt").open("w", encoding="utf-8") as text_file:
            for frame_number, (image_hw, boxes) in enumerate(zip(image_sizes, detections, strict=True), 1):
                # Each tracker receives a fresh copy of the same detector boxes.
                tracks = tracker.update(boxes.copy(), image_hw, image_hw)
                total_rows += record_frame(tracks, frame_number, text_file, min_box_area)

        keys = ("predicted_occluded_tracks", "observed_occluded_tracks",
                "oao_solver_calls", "bam_updates", "bam_damped_updates")
        stats = getattr(tracker, "oa_stats", Counter())
        summary[mode] = {"track_rows": total_rows, **{key: stats[key] for key in keys}}
        print(f"{mode}: {summary[mode]}", flush=True)
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--repo", required=True, type=Path)
    parser.add_argument("--frames", required=True, type=Path)
    parser.add_argument("--weights", required=True, type=Path)
    parser.add_argument("--config", required=True, type=Path)
    parser.add_argument("--output", required=True, type=Path)
    parser.add_argument("--sequence", required=True)
    parser.add_argument("--tau", type=float, default=0.15)
    parser.add_argument("--depth-pixels", type=float, default=5)
    parser.add_argument("--fp16", action="store_true")
    parser.add_argument("--no-gaussian", action="store_true")
    options = parser.parse_args()

    frames = sorted(path.resolve() for path in options.frames.iterdir()
                    if path.suffix.lower() in IMAGE_SUFFIXES)
    if not frames:
        raise FileNotFoundError(f"No input images at {options.frames}")
    config = OAConfig(tau=options.tau, depth_pixels=options.depth_pixels,
                      gaussian=not options.no_gaussian)
    options.output.mkdir(parents=True, exist_ok=True)

    base = importlib.import_module("trackers.hybrid_sort_tracker.hybrid_sort")
    kf = importlib.import_module("trackers.hybrid_sort_tracker.kalmanfilter_score_new")
    detections, image_sizes, tracker_spec = collect_detections(
        options.repo, frames, options.weights, options.fp16, options.config, options.output / "demo_tmp",
    )
    summary = replay(
        image_sizes, detections, tracker_spec, base, kf.KalmanFilterNew_score_new,
        ("hybrid", "oa_sort"), config, options.output, options.sequence,
    )
    report = {"spatial_metric": "Height_Modulated_IoU", "use_byte": True,
              "processed_frames": len(frames),
              "frames_without_detections": sum(len(boxes) == 0 for boxes in detections),
              "modes": summary}
    (options.output / "tracking_summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")


if __name__ == "__main__":
    main()
'''
(CODE_DIR / "oa_sort_val_runner.py").write_text(RUNNER_SOURCE, encoding="utf-8")
EVAL_SOURCE = r'''
import json
from pathlib import Path
import sys

import numpy as np
import trackeval

work = Path(sys.argv[1])
modes = ["hybrid", "oa_sort"]
manifest = json.loads((work / "run_manifest.json").read_text(encoding="utf-8"))
dataset_cfg = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
dataset_cfg.update(
    GT_FOLDER=str(work / "gt"),
    TRACKERS_FOLDER=str(work / "trackers"),
    OUTPUT_FOLDER=str(work / "trackeval"),
    TRACKERS_TO_EVAL=modes,
    TRACKER_SUB_FOLDER="data",
    CLASSES_TO_EVAL=["pedestrian"],
    BENCHMARK="DanceTrack",
    SPLIT_TO_EVAL="val",
    SKIP_SPLIT_FOL=True,
    SEQMAP_FILE=str(work / "seqmap.txt"),
    PRINT_CONFIG=False,
)
eval_cfg = trackeval.Evaluator.get_default_eval_config()
eval_cfg.update(
    USE_PARALLEL=False, PRINT_CONFIG=False, PRINT_RESULTS=False,
    OUTPUT_SUMMARY=True, OUTPUT_DETAILED=True, PLOT_CURVES=False,
    TIME_PROGRESS=False, RETURN_ON_ERROR=True,
)
dataset = trackeval.datasets.MotChallenge2DBox(dataset_cfg)
metrics = [
    trackeval.metrics.HOTA(),
    trackeval.metrics.CLEAR({"THRESHOLD": 0.5, "PRINT_CONFIG": False}),
    trackeval.metrics.Identity({"THRESHOLD": 0.5, "PRINT_CONFIG": False}),
]
results, messages = trackeval.Evaluator(eval_cfg).evaluate([dataset], metrics)


def metric_values(result):
    hota, clear, identity = (result[key] for key in ("HOTA", "CLEAR", "Identity"))
    return {
        "HOTA": 100 * float(np.mean(hota["HOTA"])),
        "AssA": 100 * float(np.mean(hota["AssA"])),
        "DetA": 100 * float(np.mean(hota["DetA"])),
        "IDF1": 100 * float(identity["IDF1"]),
        "MOTA": 100 * float(clear["MOTA"]),
        "IDSW": int(clear["IDSW"]),
        "FP": int(clear["CLR_FP"]),
        "FN": int(clear["CLR_FN"]),
        "Frag": int(clear["Frag"]),
    }


dataset_name = dataset.get_name()
report = {}
for mode in modes:
    mode_results = results[dataset_name][mode]
    if mode_results is None:
        raise RuntimeError(f"TrackEval {mode}: {messages[dataset_name][mode]}")
    report[mode] = {
        sequence: metric_values(mode_results[sequence]["pedestrian"])
        for sequence in ("COMBINED_SEQ", *manifest["sequence_frames"])
    }
(work / "metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
'''
(CODE_DIR / "evaluate_trackeval.py").write_text(EVAL_SOURCE, encoding="utf-8")


## 6. Hybrid-SORT와 OA-SORT 실행

검출 입력을 메모리에서 공유하고 MOT 형식의 결과를 저장합니다. 시퀀스마다 완료 여부와 파일 해시를 저장하므로 같은 설정으로 다시 실행하면 완료한 시퀀스는 건너뜁니다. 중간에 실패하면 해당 시퀀스의 로그 마지막 부분이 표시됩니다.


In [ ]:
import json
import tempfile
import time

MODES = ("hybrid", "oa_sort")
SOURCE_FILES = [CODE_DIR / "oa_sort_impl.py", CODE_DIR / "oa_sort_val_runner.py"]
manifest = {
    "scope": "smoke" if SMOKE_SEQUENCE else "val",
    "official_val_seqmap_sha256": digest_file(official_map),
    "sequence_frames": {name: len(DATA[name]["frames"]) for name in SEQUENCES},
    "hybrid_sort_revision": REVISION,
    "detector_config": VAL_CFG,
    "detector_config_sha256": digest_file(REPO / VAL_CFG),
    "checkpoint_name": checkpoint.name,
    "checkpoint_sha256": checkpoint_sha256,
    "legacy_checkpoint": ALLOW_LEGACY_CHECKPOINT,
    "fp16": USE_FP16,
    "oa": {"tau": OA_TAU, "depth_pixels": OA_DEPTH_PIXELS, "gaussian": OA_GAUSSIAN,
           "kx": 3 * 2 ** 0.5, "ky": 3, "final_gate": "Hybrid-SORT original threshold"},
    "code_sha256": {path.name: digest_file(path) for path in SOURCE_FILES},
    "data": {
        name: {
            "seqinfo_sha256": hashlib.sha256(DATA[name]["info"]).hexdigest(),
            "gt_sha256": hashlib.sha256(DATA[name]["gt"]).hexdigest(),
            "image_inventory_sha256": hashlib.sha256(
                json.dumps(DATA[name]["inventory"], separators=(",", ":")).encode()
            ).hexdigest(),
        }
        for name in SEQUENCES
    },
}
fingerprint = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
BENCH_DIR = Path(RESULTS_ROOT).expanduser() / f"{manifest['scope']}-{fingerprint}"
GT_ROOT = BENCH_DIR / "gt"
TRACKERS_ROOT = BENCH_DIR / "trackers"
LOG_DIR = BENCH_DIR / "logs"
STATUS_DIR = BENCH_DIR / "completed"
for directory in [GT_ROOT, LOG_DIR, STATUS_DIR, *[TRACKERS_ROOT / mode / "data" for mode in MODES]]:
    directory.mkdir(parents=True, exist_ok=True)

(BENCH_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
SEQMAP_FILE = BENCH_DIR / "seqmap.txt"
SEQMAP_FILE.write_text("name\n" + "\n".join(SEQUENCES) + "\n", encoding="utf-8")

for name in SEQUENCES:
    gt_sequence = GT_ROOT / name
    (gt_sequence / "gt").mkdir(parents=True, exist_ok=True)
    (gt_sequence / "seqinfo.ini").write_bytes(DATA[name]["info"])
    (gt_sequence / "gt" / "gt.txt").write_bytes(DATA[name]["gt"])


def result_path(mode, name):
    return TRACKERS_ROOT / mode / "data" / f"{name}.txt"


def is_complete(name):
    marker = STATUS_DIR / f"{name}.json"
    if not marker.is_file():
        return False
    try:
        state = json.loads(marker.read_text(encoding="utf-8"))
        return state["processed_frames"] == len(DATA[name]["frames"]) and all(
            result_path(mode, name).is_file()
            and state["mot_sha256"][mode] == digest_file(result_path(mode, name))
            for mode in MODES
        )
    except (OSError, ValueError, KeyError):
        return False


def run_sequence(name, work):
    details = DATA[name]
    if VAL_DIR is None:
        frames_dir = work / "img1"
        frames_dir.mkdir()
        with ZipFile(ZIP_PATH) as archive:
            for name_in_zip, image in zip(details["entries"], details["frames"], strict=True):
                with archive.open(name_in_zip) as stream, (frames_dir / image).open("wb") as out:
                    shutil.copyfileobj(stream, out)
    else:
        frames_dir = VAL_DIR / name / "img1"

    destination = work / "tracking"
    command = [PYTHON, CODE_DIR / "oa_sort_val_runner.py", "--repo", REPO,
               "--frames", frames_dir, "--weights", checkpoint, "--config", REPO / VAL_CFG,
               "--output", destination, "--sequence", name, "--tau", OA_TAU,
               "--depth-pixels", OA_DEPTH_PIXELS]
    if USE_FP16:
        command.append("--fp16")
    if not OA_GAUSSIAN:
        command.append("--no-gaussian")
    tracking_env = {**BASE_ENV, "PYTHONUNBUFFERED": "1"}
    if ALLOW_LEGACY_CHECKPOINT:
        tracking_env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

    log_path = LOG_DIR / f"{name}.log"
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.run([str(arg) for arg in command], cwd=REPO, env=tracking_env,
                                 stdout=log, stderr=subprocess.STDOUT)
    if process.returncode:
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-35:]
        raise RuntimeError(f"{name} 실패 (exit={process.returncode})\n"
                           + "\n".join(tail) + f"\n로그 {log_path}")

    report = json.loads((destination / "tracking_summary.json").read_text(encoding="utf-8"))
    if report["processed_frames"] != len(details["frames"]):
        raise RuntimeError(f"{name}: 저장된 추적 프레임 수가 seqinfo와 다릅니다.")

    report["mot_sha256"] = {}
    for mode in MODES:
        target = result_path(mode, name)
        partial = target.with_suffix(".txt.part")
        shutil.copyfile(destination / mode / f"{name}.txt", partial)
        report["mot_sha256"][mode] = digest_file(partial)
        partial.replace(target)
    marker = STATUS_DIR / f"{name}.json"
    temporary = marker.with_suffix(".json.part")
    temporary.write_text(json.dumps(report, indent=2), encoding="utf-8")
    temporary.replace(marker)
    return report


print("결과 폴더", BENCH_DIR)
started = time.monotonic()
for index, name in enumerate(SEQUENCES, 1):
    if is_complete(name):
        print(f"[{index}/{len(SEQUENCES)}] {name} · 기존 결과 사용")
        continue
    print(f"[{index}/{len(SEQUENCES)}] {name} · {len(DATA[name]['frames']):,}프레임 추적 중", flush=True)
    with tempfile.TemporaryDirectory(prefix=f"{name}-", dir=ROOT) as temporary:
        report = run_sequence(name, Path(temporary))
    print(f"  Hybrid {report['modes']['hybrid']['track_rows']:,}행 · "
          f"OA {report['modes']['oa_sort']['track_rows']:,}행 · "
          f"누적 {(time.monotonic() - started) / 60:.1f}분", flush=True)

print(f"{len(SEQUENCES)}개 시퀀스 완료")


## 7. 전체 및 시퀀스별 TrackEval 평가

전체 실행에서는 `COMBINED`에 논문 표 11의 validation 참고값을 나란히 표시합니다. TrackEval이 시퀀스 전체를 합산한 값이며, 시퀀스별 점수는 CSV에서 확인할 수 있습니다.


In [ ]:
import csv
from IPython.display import Markdown, display

missing = [name for name in SEQUENCES if not is_complete(name)]
if missing:
    raise RuntimeError("추적 셀을 먼저 완료하세요. 누락: " + ", ".join(missing))

TRACKEVAL_VERSION = "1.0.0"
eval_venv = ROOT / f"trackeval_venv_{TRACKEVAL_VERSION}"
eval_python = eval_venv / "bin" / "python"
installed = eval_venv / ".trackeval-installed"
if not eval_python.is_file():
    run([uv, "venv", "--python", "3.11", eval_venv])
if not installed.is_file():
    run([uv, "pip", "install", "--python", eval_python, f"trackeval=={TRACKEVAL_VERSION}"])
    installed.touch()
eval_env = {**os.environ, "PYTHONPATH": "", "MPLBACKEND": "Agg"}
run([eval_python, CODE_DIR / "evaluate_trackeval.py", BENCH_DIR], env=eval_env)

metrics_by_mode = json.loads((BENCH_DIR / "metrics.json").read_text(encoding="utf-8"))
count_metrics = {"IDSW", "FP", "FN", "Frag"}
paper = {
    "HOTA": (59.39, 61.49),
    "AssA": (44.90, 48.03),
    "IDF1": (60.67, 63.65),
}
records = []
for name in ("COMBINED_SEQ", *SEQUENCES):
    hybrid, oa = (metrics_by_mode[mode][name] for mode in MODES)
    for metric, original in hybrid.items():
        reference = paper.get(metric) if name == "COMBINED_SEQ" and not SMOKE_SEQUENCE else None
        row = {
            "sequence": "COMBINED" if name == "COMBINED_SEQ" else name,
            "metric": metric, "unit": "count" if metric in count_metrics else "percent",
            "hybrid": original, "oa_sort": oa[metric],
            "oa_minus_hybrid": oa[metric] - original,
            "paper_hybrid": reference[0] if reference else None,
            "paper_oa_sort": reference[1] if reference else None,
            "hybrid_minus_paper": original - reference[0] if reference else None,
            "oa_minus_paper": oa[metric] - reference[1] if reference else None,
        }
        records.append(row)

csv_path = BENCH_DIR / "oa_sort_val_metrics.csv"
with csv_path.open("w", encoding="utf-8-sig", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)

label = f"{SMOKE_SEQUENCE} · 단일 영상 점검" if SMOKE_SEQUENCE else "DanceTrack val 전체 · COMBINED"
lines = [
    f"### {label}",
    "",
    "| 지표 | Hybrid-SORT | OA-SORT 재구현 | OA − Hybrid | 논문 Hybrid | 논문 OA |",
    "|---|---:|---:|---:|---:|---:|",
]
for row in records:
    if row["sequence"] != "COMBINED":
        continue
    metric = row["metric"]
    hybrid, oa, difference = (row[key] for key in ("hybrid", "oa_sort", "oa_minus_hybrid"))
    if metric in count_metrics:
        measured = f"{hybrid:d} | {oa:d} | {difference:+d} 건"
    else:
        measured = f"{hybrid:.2f}% | {oa:.2f}% | {difference:+.2f} %p"
    reference = (f"{row['paper_hybrid']:.2f}% | {row['paper_oa_sort']:.2f}%"
                 if row["paper_hybrid"] is not None else "— | —")
    lines.append(f"| {metric} | {measured} | {reference} |")
display(Markdown("\n".join(lines)))
print("CSV (전체 및 시퀀스별)", csv_path)
print("TrackEval 원본 출력", BENCH_DIR / "trackeval")


## 8. 결과 ZIP 다운로드

비교 CSV, TrackEval 원본 리포트, 추적 MOT TXT, 설정 및 실행 로그를 ZIP으로 다운로드합니다.


In [ ]:
from zipfile import ZIP_DEFLATED

archive_path = BENCH_DIR / "oa_sort_val_results.zip"
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in ("run_manifest.json", "seqmap.txt", "metrics.json", "oa_sort_val_metrics.csv"):
        archive.write(BENCH_DIR / name, name)
    for folder in ("trackers", "trackeval", "completed", "logs"):
        for path in sorted((BENCH_DIR / folder).rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(BENCH_DIR))
    for path in (*SOURCE_FILES, CODE_DIR / "evaluate_trackeval.py"):
        archive.write(path, Path("source") / path.name)

print("결과 ZIP", archive_path)
from google.colab import files
files.download(str(archive_path))
